In [1]:
import os
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_google_genai import ChatGoogleGenerativeAI

C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
API_KEY = 'API_KEY'

In [3]:
llm = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    google_api_key = API_KEY,
    temperature = 0.2
)

In [4]:
class FinanceState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
 
    # Applicant inputs
    applicant_name: str
    personal_details: str
    salary_info: str
    bank_statements: str
    loan_amount_requested: float
    employment_type: str
    credit_score: int
 
    # Stage 1 — Applicant Analysis
    income_stability: str
    missing_documents: str
    spending_patterns: str
    financial_behavior_score: str
 
    # Stage 2 — Verification
    credit_bureau_result: str
    fraud_detection_result: str
    kyc_result: str
    employment_verification_result: str
    debt_to_income_ratio: str
 
    # Stage 3 — Routing
    routed_to: str
 
    # Stage 4 — Decision Orchestration
    approval_rationale: str
    risk_summary: str
    recommended_loan_amount: str
    interest_slab: str
    applicant_notification: str
    final_decision_report: str

In [5]:
def applicant_analysis_node(state: FinanceState) -> dict:
    prompt = f"""You are a financial analyst.
Applicant: {state.get('applicant_name')}
Employment type: {state.get('employment_type')}
Salary: {state.get('salary_info')}
Bank statements: {state.get('bank_statements')}
Personal details: {state.get('personal_details')}
Loan requested: {state.get('loan_amount_requested')}

Respond in EXACTLY this format (no extra text):
income_stability: <one of: stable | unstable | variable>
financial_behavior_score: <one of: excellent | good | average | poor>
spending_patterns: <one of: conservative | moderate | excessive>
missing_documents: <comma-separated list of missing docs or 'none'>"""

    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    parsed = {k.strip(): v.strip() for k, v in (line.split(":", 1) for line in response.splitlines() if ":" in line)}

    return {
        "income_stability": parsed.get("income_stability", "variable"),
        "financial_behavior_score": parsed.get("financial_behavior_score", "average"),
        "spending_patterns": parsed.get("spending_patterns", "moderate"),
        "missing_documents": parsed.get("missing_documents", "none"),
        "messages": [AIMessage(content=f"[Applicant Analysis] {parsed}")],
    }

In [6]:
def verification_node(state: FinanceState) -> dict:
    credit_score = state.get("credit_score", 650)

    if credit_score >= 800:
        bureau = f"Score {credit_score}: EXCELLENT. Clean history."
    elif credit_score >= 700:
        bureau = f"Score {credit_score}: GOOD. Minor delinquency."
    elif credit_score >= 600:
        bureau = f"Score {credit_score}: FAIR. 2 missed payments."
    else:
        bureau = f"Score {credit_score}: POOR. Multiple defaults."

    prompt = f"""You are a bank verification system.
Applicant: {state.get('applicant_name')}
Personal details: {state.get('personal_details')}
Employment: {state.get('employment_type')}
Salary: {state.get('salary_info')}
Bank statements: {state.get('bank_statements')}
Loan requested: {state.get('loan_amount_requested')}

Respond in EXACTLY this format (no extra text):
fraud_detection_result: <one of: clean | low_risk | medium_risk | high_risk>
kyc_result: <one of: passed | incomplete>
employment_verification_result: <one of: verified | partial | unverified>
debt_to_income_ratio: <percentage e.g. 35%>
dti_assessment: <one of: healthy | borderline | excessive>"""

    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    parsed = {k.strip(): v.strip() for k, v in (line.split(":", 1) for line in response.splitlines() if ":" in line)}

    return {
        "credit_bureau_result": bureau,
        "fraud_detection_result": parsed.get("fraud_detection_result", "low_risk"),
        "kyc_result": parsed.get("kyc_result", "passed"),
        "employment_verification_result": parsed.get("employment_verification_result", "verified"),
        "debt_to_income_ratio": f"{parsed.get('debt_to_income_ratio', '40%')} ({parsed.get('dti_assessment', 'borderline')})",
        "messages": [AIMessage(content=f"[Verification] {parsed}")],
    }

In [7]:
def loan_routing_node(state: FinanceState) -> dict:
    """Determine which agent handles the loan case."""
    credit_score = state.get("credit_score", 650)
    fraud_result = state.get("fraud_detection_result", "").lower()
    missing_docs = state.get("missing_documents", "none").lower()
    dti = state.get("debt_to_income_ratio", "").lower()
    kyc = state.get("kyc_result", "").lower()
    financial_behavior = state.get("financial_behavior_score", "average").lower()
 
    # Priority ordering matters
    if "high_risk" in fraud_result:
        routed_to = "fraud_investigation_agent"
    elif missing_docs != "none" or "incomplete" in kyc:
        routed_to = "clarification_agent"
    elif credit_score >= 780 and financial_behavior in ("excellent", "good") and "healthy" in dti:
        routed_to = "fast_track_approval_agent"
    elif "excessive" in dti or credit_score < 600:
        routed_to = "rejection_recommendation_agent"
    else:
        routed_to = "manual_review_agent"
 
    return {
        "routed_to": routed_to,
        "messages": [AIMessage(content=f"[Loan Router] Routing to → {routed_to}")],
    }
 
 
def loan_router_edge(state: FinanceState) -> Literal[
    "fraud_investigation_agent", "fast_track_approval_agent",
    "clarification_agent", "manual_review_agent", "rejection_recommendation_agent"
]:
    return state["routed_to"]

In [8]:
def _loan_agent_assessment(role: str, context: dict) -> str:
    prompt = f"""You are the {role} at a financial institution.
Context:
{chr(10).join(f'  {k}: {v}' for k, v in context.items())}
 
Provide your assessment and recommended next steps (3-4 sentences)."""
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()
 
 
def fraud_investigation_agent_node(state: FinanceState) -> dict:
    assessment = _loan_agent_assessment("Fraud Investigation Officer", {
        "applicant": state.get("applicant_name"),
        "fraud_detection": state.get("fraud_detection_result"),
        "kyc": state.get("kyc_result"),
        "personal_details": state.get("personal_details"),
    })
    return {"routed_to": "fraud_investigation_agent",
            "messages": [AIMessage(content=f"[Fraud Investigation Agent] {assessment}")]}
 
 
def fast_track_approval_agent_node(state: FinanceState) -> dict:
    assessment = _loan_agent_assessment("Fast-Track Loan Approval Officer", {
        "applicant": state.get("applicant_name"),
        "credit_score": state.get("credit_score"),
        "income_stability": state.get("income_stability"),
        "loan_requested": state.get("loan_amount_requested"),
        "dti": state.get("debt_to_income_ratio"),
    })
    return {"routed_to": "fast_track_approval_agent",
            "messages": [AIMessage(content=f"[Fast-Track Agent] {assessment}")]}
 
 
def clarification_agent_node(state: FinanceState) -> dict:
    assessment = _loan_agent_assessment("Document Clarification Agent", {
        "applicant": state.get("applicant_name"),
        "missing_documents": state.get("missing_documents"),
        "kyc_result": state.get("kyc_result"),
    })
    return {"routed_to": "clarification_agent",
            "messages": [AIMessage(content=f"[Clarification Agent] {assessment}")]}
 
 
def manual_review_agent_node(state: FinanceState) -> dict:
    assessment = _loan_agent_assessment("Senior Credit Analyst (Manual Review)", {
        "applicant": state.get("applicant_name"),
        "credit_score": state.get("credit_score"),
        "income_stability": state.get("income_stability"),
        "dti": state.get("debt_to_income_ratio"),
        "financial_behavior": state.get("financial_behavior_score"),
        "spending_patterns": state.get("spending_patterns"),
    })
    return {"routed_to": "manual_review_agent",
            "messages": [AIMessage(content=f"[Manual Review Agent] {assessment}")]}
 
 
def rejection_recommendation_agent_node(state: FinanceState) -> dict:
    assessment = _loan_agent_assessment("Loan Rejection Recommendation Officer", {
        "applicant": state.get("applicant_name"),
        "credit_score": state.get("credit_score"),
        "dti": state.get("debt_to_income_ratio"),
        "fraud_result": state.get("fraud_detection_result"),
        "financial_behavior": state.get("financial_behavior_score"),
    })
    return {"routed_to": "rejection_recommendation_agent",
            "messages": [AIMessage(content=f"[Rejection Agent] {assessment}")]}

In [9]:
def approval_rationale_node(state: FinanceState) -> dict:
    prompt = f"""Generate an approval/rejection rationale for this loan application.
Applicant: {state.get('applicant_name')}
Credit Score: {state.get('credit_score')}
Income Stability: {state.get('income_stability')}
Financial Behavior: {state.get('financial_behavior_score')}
Fraud Result: {state.get('fraud_detection_result')}
KYC: {state.get('kyc_result')}
DTI: {state.get('debt_to_income_ratio')}
Routing Decision: {state.get('routed_to')}
Missing Docs: {state.get('missing_documents')}
 
Write a 3-4 sentence formal rationale for the loan decision."""
 
    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return {
        "approval_rationale": response,
        "messages": [AIMessage(content=f"[Approval Rationale]\n{response}")],
    }
 
 
def risk_summary_node(state: FinanceState) -> dict:
    prompt = f"""Create a concise risk summary for the underwriting team.
Applicant: {state.get('applicant_name')}
Credit Bureau: {state.get('credit_bureau_result')}
Fraud: {state.get('fraud_detection_result')}
Employment: {state.get('employment_verification_result')}
DTI: {state.get('debt_to_income_ratio')}
Spending: {state.get('spending_patterns')}
 
Provide a bullet-point risk summary (4-5 bullets)."""
 
    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return {
        "risk_summary": response,
        "messages": [AIMessage(content=f"[Risk Summary]\n{response}")],
    }
 
 
def loan_amount_recommendation_node(state: FinanceState) -> dict:
    """Recommend actual loan amount based on profile."""
    routed_to = state.get("routed_to", "manual_review_agent")
    requested = state.get("loan_amount_requested", 0)
    credit_score = state.get("credit_score", 650)
    dti = state.get("debt_to_income_ratio", "40%")
 
    if routed_to == "rejection_recommendation_agent" or routed_to == "fraud_investigation_agent":
        recommended = "₹0 (Application Rejected / Under Investigation)"
    elif routed_to == "fast_track_approval_agent":
        recommended = f"₹{requested:,.0f} (Full amount approved)"
    elif routed_to == "clarification_agent":
        recommended = "On hold — pending document submission"
    elif credit_score >= 700:
        recommended = f"₹{min(requested, requested * 0.85):,.0f} (85% of requested amount)"
    else:
        recommended = f"₹{min(requested, requested * 0.60):,.0f} (60% of requested — reduced due to risk)"
 
    return {
        "recommended_loan_amount": recommended,
        "messages": [AIMessage(content=f"[Loan Amount] Recommended: {recommended}")],
    }
 
 
def interest_slab_node(state: FinanceState) -> dict:
    """Suggest interest rate slab."""
    credit_score = state.get("credit_score", 650)
    routed_to = state.get("routed_to", "manual_review_agent")
 
    if routed_to in ("rejection_recommendation_agent", "fraud_investigation_agent"):
        slab = "N/A — Application not approved"
    elif credit_score >= 800:
        slab = "8.5% p.a. (Prime Rate — Excellent Profile)"
    elif credit_score >= 750:
        slab = "10.0% p.a. (Standard Rate — Good Profile)"
    elif credit_score >= 700:
        slab = "11.5% p.a. (Moderate Rate — Average Profile)"
    else:
        slab = "13.5% p.a. (High-Risk Rate — if approved)"
 
    return {
        "interest_slab": slab,
        "messages": [AIMessage(content=f"[Interest Slab] {slab}")],
    }
 
 
def applicant_notification_node(state: FinanceState) -> dict:
    """Generate the applicant-facing notification."""
    routed_to = state.get("routed_to", "manual_review_agent")
    name = state.get("applicant_name", "Applicant")
 
    status_messages = {
        "fast_track_approval_agent": f"Congratulations {name}! Your loan application has been approved on a fast-track basis. Disbursement will be processed within 24 hours.",
        "fraud_investigation_agent": f"Dear {name}, your application is currently under additional review for security verification. Our team will contact you within 2-3 business days.",
        "clarification_agent": f"Dear {name}, we require additional documents to process your application. Please submit the missing documents at your earliest convenience.",
        "manual_review_agent": f"Dear {name}, your loan application is under review by our credit team. You will receive a decision within 3-5 business days.",
        "rejection_recommendation_agent": f"Dear {name}, after careful evaluation, we are unable to approve your loan application at this time. You may reapply after 6 months or improving your credit profile.",
    }
 
    notification = status_messages.get(routed_to, f"Dear {name}, your application is being processed.")
    return {
        "applicant_notification": notification,
        "messages": [AIMessage(content=f"[Applicant Notification] {notification}")],
    }

In [10]:
def final_decision_report_node(state: FinanceState) -> dict:
    """Compile full decision report."""
    report = f"""
========== LOAN RISK ASSESSMENT & DECISION REPORT ==========
Applicant Name      : {state.get('applicant_name')}
Loan Amount Req.    : ₹{state.get('loan_amount_requested'):,.0f}
Employment Type     : {state.get('employment_type')}
Credit Score        : {state.get('credit_score')}
 
--- Stage 1: Applicant Analysis ---
Income Stability    : {state.get('income_stability')}
Spending Patterns   : {state.get('spending_patterns')}
Financial Behavior  : {state.get('financial_behavior_score')}
Missing Documents   : {state.get('missing_documents')}
 
--- Stage 2: Verification Results ---
Credit Bureau       : {state.get('credit_bureau_result')}
Fraud Detection     : {state.get('fraud_detection_result')}
KYC                 : {state.get('kyc_result')}
Employment          : {state.get('employment_verification_result')}
Debt-to-Income      : {state.get('debt_to_income_ratio')}
 
--- Stage 3: Routing ---
Routed To           : {state.get('routed_to')}
 
--- Stage 4: Decision ---
Recommended Amount  : {state.get('recommended_loan_amount')}
Interest Slab       : {state.get('interest_slab')}
 
Approval Rationale  :
{state.get('approval_rationale')}
 
Risk Summary        :
{state.get('risk_summary')}
 
Applicant Message   :
{state.get('applicant_notification')}
============================================================="""
 
    print(report)
    return {
        "final_decision_report": report,
        "messages": [AIMessage(content=report)],
    }

In [11]:
def build_finance_graph() -> StateGraph:
    graph = StateGraph(FinanceState)

    # Stage 1 — single combined analysis node
    graph.add_node("applicant_analysis", applicant_analysis_node)

    # Stage 2 — single combined verification node
    graph.add_node("verification", verification_node)

    # Stage 3 — routing
    graph.add_node("loan_routing", loan_routing_node)
    graph.add_node("fraud_investigation_agent", fraud_investigation_agent_node)
    graph.add_node("fast_track_approval_agent", fast_track_approval_agent_node)
    graph.add_node("clarification_agent", clarification_agent_node)
    graph.add_node("manual_review_agent", manual_review_agent_node)
    graph.add_node("rejection_recommendation_agent", rejection_recommendation_agent_node)

    # Stage 4 — parallel decision generation
    graph.add_node("approval_rationale", approval_rationale_node)
    graph.add_node("risk_summary", risk_summary_node)
    graph.add_node("loan_amount_recommendation", loan_amount_recommendation_node)
    graph.add_node("interest_slab", interest_slab_node)
    graph.add_node("applicant_notification", applicant_notification_node)
    graph.add_node("final_decision_report", final_decision_report_node)

    # START → Stage 1
    graph.add_edge(START, "applicant_analysis")

    # Stage 1 → Stage 2
    graph.add_edge("applicant_analysis", "verification")

    # Stage 2 → Routing
    graph.add_edge("verification", "loan_routing")

    # Conditional routing
    graph.add_conditional_edges(
        "loan_routing",
        loan_router_edge,
        {
            "fraud_investigation_agent": "fraud_investigation_agent",
            "fast_track_approval_agent": "fast_track_approval_agent",
            "clarification_agent": "clarification_agent",
            "manual_review_agent": "manual_review_agent",
            "rejection_recommendation_agent": "rejection_recommendation_agent",
        },
    )

    # Specialist agents → Stage 4
    specialist_agents = [
        "fraud_investigation_agent", "fast_track_approval_agent",
        "clarification_agent", "manual_review_agent", "rejection_recommendation_agent",
    ]
    stage4_nodes = ["approval_rationale", "risk_summary", "loan_amount_recommendation",
                    "interest_slab", "applicant_notification"]

    for agent in specialist_agents:
        for s4 in stage4_nodes:
            graph.add_edge(agent, s4)

    # Stage 4 → Final Report
    for s4 in stage4_nodes:
        graph.add_edge(s4, "final_decision_report")

    graph.add_edge("final_decision_report", END)

    return graph.compile()

In [12]:
def run_finance_system(applicant: dict):
    print(f"APPLICANT: {applicant.get('applicant_name')} | Loan: ₹{applicant.get('loan_amount_requested', 0):,.0f}")

    app = build_finance_graph()
    initial_state: FinanceState = {
        "messages": [HumanMessage(content=f"Loan application from {applicant.get('applicant_name')}")],
        "applicant_name": applicant.get("applicant_name", "Unknown"),
        "personal_details": applicant.get("personal_details", ""),
        "salary_info": applicant.get("salary_info", ""),
        "bank_statements": applicant.get("bank_statements", ""),
        "loan_amount_requested": float(applicant.get("loan_amount_requested", 0)),
        "employment_type": applicant.get("employment_type", "salaried"),
        "credit_score": int(applicant.get("credit_score", 650)),
        "income_stability": "",
        "missing_documents": "",
        "spending_patterns": "",
        "financial_behavior_score": "",
        "credit_bureau_result": "",
        "fraud_detection_result": "",
        "kyc_result": "",
        "employment_verification_result": "",
        "debt_to_income_ratio": "",
        "routed_to": "",
        "approval_rationale": "",
        "risk_summary": "",
        "recommended_loan_amount": "",
        "interest_slab": "",
        "applicant_notification": "",
        "final_decision_report": "",
    }
 
    result = app.invoke(initial_state)
    return result["final_decision_report"]

In [13]:
if __name__ == "__main__":
    applicants = [
        {
            "applicant_name": "Arjun Sharma",
            "personal_details": "Age 32, PAN ABCDE1234F, Aadhaar 1234-5678-9012, Mumbai. ID and address proof submitted.",
            "salary_info": "Monthly salary ₹85,000. Working at TCS for 6 years.",
            "bank_statements": "Regular salary credits. Monthly EMIs ₹12,000. Occasional large transfers.",
            "loan_amount_requested": 1500000,
            "employment_type": "salaried",
            "credit_score": 810,
        }
    ]
 
    for applicant in applicants:
        print(run_finance_system(applicant))
        print("\n")

APPLICANT: Arjun Sharma | Loan: ₹1,500,000

========== LOAN RISK ASSESSMENT & DECISION REPORT ==========
Applicant Name      : Arjun Sharma
Loan Amount Req.    : ₹1,500,000
Employment Type     : salaried
Credit Score        : 810

--- Stage 1: Applicant Analysis ---
Income Stability    : stable
Spending Patterns   : conservative
Financial Behavior  : good
Missing Documents   : payslips, IT returns

--- Stage 2: Verification Results ---
Credit Bureau       : Score 810: EXCELLENT. Clean history.
Fraud Detection     : low_risk
KYC                 : passed
Employment          : verified
Debt-to-Income      : 14% (healthy)

--- Stage 3: Routing ---
Routed To           : clarification_agent

--- Stage 4: Decision ---
Recommended Amount  : On hold — pending document submission
Interest Slab       : 8.5% p.a. (Prime Rate — Excellent Profile)

Approval Rationale  :
Mr. Arjun Sharma's loan application demonstrates a robust financial standing, characterized by an excellent credit score of 810, st